In [0]:
%sql

-- CASTEANDO CADA COLUMNA A SU TIPO DE DATO ESPERADO
SELECT 
    CAST(id_transaccion AS STRING),
    CAST(fecha AS DATE),
    CAST(id_cliente AS STRING),
    CAST(categoria_producto AS STRING),
    CAST(cantidad AS DOUBLE),
    CAST(precio_unitario AS DOUBLE),
    CAST(metodo_pago AS STRING),
    CAST(region AS STRING),
    CAST(satisfaccion_cliente AS DOUBLE)
FROM workspace.electiva.ventas 

In [0]:
%sql

select 
    try_cast(id_cliente as int) 
from  workspace.electiva.ventas 

In [0]:
%sql
SELECT 
   TRY_CAST(ID_CLIENTE AS INT)
from workspace.electiva.ventas 

In [0]:
%sql
select 
    * 
from workspace.electiva.ventas
where cantidad = null; -- al comparar cantidad = null no aparece ningun registro, lo cual es un resultado incorrecto

In [0]:
%sql
select 
    * 
from workspace.electiva.ventas
where cantidad is   null; -- al comparar cantidad  is null nos muestra los registros que no tienen un valor en la columna cantidad, que es el resultado esperado correcto

In [0]:
%sql
-- 1. Crear un Volumen en Unity Catalog (ejemplo dentro del esquema actual)
CREATE VOLUME IF NOT EXISTS workspace.electiva.mi_volumen_datos
COMMENT 'Volumen para almacenamiento de archivos en bruto';    

--1.1 Subir archivo ventas_raw.csv

In [0]:
## 1.2 Ejecutar esto para transformar el csv a parquet 
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DecimalType, DateType

# 1.3 Definir el esquema explícito para que coincida con tu tabla Delta/Externa
esquema = StructType([
    StructField("id_transaccion", StringType(), True),
    StructField("fecha", StringType(), True),
    StructField("id_cliente", StringType(), True),
    StructField("categoria_producto", StringType(), True),
    StructField("cantidad", StringType(), True),
    StructField("precio_unitario", DecimalType(), True),
    StructField("metodo_pago", StringType(), True),
    StructField("region", StringType(), True),
    StructField("satisfaccion_cliente", StringType(), True)


])

# 1.4 Leer el CSV desde la ruta del Volumen donde subiste el archivo
df_csv = spark.read \
    .option("header", "true") \
    .schema(esquema) \
    .csv("/Volumes/workspace/electiva/mi_volumen_datos/ventas.csv")

# 1.5 Guardar en formato Parquet en la subcarpeta del Volumen destinada a la Tabla Externa
df_csv.write \
    .format("parquet") \
    .mode("append") \
    .save("/Volumes/workspace/electiva/mi_volumen_datos/tabla_ventas_parquet")

In [0]:
%sql
-- 2. Consultar datos directamente desde un archivo Parquet o CSV almacenado en el Volumen
SELECT * 
FROM parquet.`/Volumes/workspace/electiva/mi_volumen_datos/tabla_ventas_parquet/`;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.electiva.tabla_ventas_desde_volumen
USING DELTA
AS SELECT * 
FROM parquet.`/Volumes/workspace/electiva/mi_volumen_datos/tabla_ventas_parquet/`;

In [0]:
%sql
SELECT * FROM workspace.electiva.tabla_ventas_desde_volumen;